# 🏛️ Fine-Tuning vilm/vinallama-7b-chat trên Google Colab T4 Free
### Dự án: Trợ lý AI Tư vấn Pháp luật An toàn Giao thông đường bộ Việt Nam

Notebook chuẩn hóa này được thiết kế tối ưu hóa 100% cho phần cứng **GPU Tesla T4 (Free Tier trên Colab)**:
- **Mô hình nền (Base Model)**: `vilm/vinallama-7b-chat` (LLaMA-2 7B tinh chỉnh tiếng Việt).
- **Tập dữ liệu huấn luyện**: 683 cặp câu hỏi - đáp pháp luật giao thông đã được chuẩn hóa theo Luật TTATGTĐB 2024 & Nghị định 168/2024/NĐ-CP.
- **Kỹ thuật huấn luyện**: **4-bit QLoRA** (tiết kiệm 70% VRAM, tiêu hao ~5.5GB / 15GB VRAM của T4).
- **Chế độ kiểm định (Validation)**: Theo dõi `eval_loss` sau mỗi epoch để chống học vẹt (overfitting).
- **Đóng gói xuất xưởng**: Hợp nhất trọng số và Lượng tử hóa sang **GGUF (Q4_K_M)** mang tên **`vinallama-7b-chat-traffic-law.Q4_K_M.gguf`**.

## Bước 1: Kiểm tra phần cứng GPU & Cài đặt môi trường thư viện
Lưu ý: Hãy đảm bảo bạn đã chọn **Runtime -> Change runtime type -> T4 GPU** trước khi chạy.

In [ ]:
# 1. Kiểm tra thông số phần cứng GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# 2. Dọn dẹp các gói gây xung đột nền của Colab (cudf, diffusers, gradio, torchao)
!pip uninstall -y -q cudf-cu13 diffusers gradio gradio-client torchao

# 3. Cài đặt các thư viện lõi (chỉ hiển thị thanh tiến trình có màu, không hiện dòng Downloading và không hiện Requirement satisfied)
!FORCE_COLOR=1 pip install -U transformers datasets peft trl bitsandbytes accelerate huggingface-hub --progress-bar on | grep --line-buffered -E "━━━|Successfully installed"

# 4. Cấu hình môi trường chuẩn hóa & dập tắt các log cảnh báo không cần thiết
import warnings, os
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print("\n>>> Toàn bộ thư viện đã được cài đặt thành công, tương thích 100% không còn xung đột!")


## Bước 2: Nạp dữ liệu huấn luyện (`train.jsonl` và `val.jsonl`)
Nếu bạn đã kéo thả 2 file vào thanh công cụ Files bên trái, ô này sẽ tự động nhận diện và nạp vào hệ thống.

In [ ]:
import os
from google.colab import files

if not os.path.exists('train.jsonl') or not os.path.exists('val.jsonl'):
    print("Chưa thấy file trong thư mục. Hãy chọn tải lên 2 file 'train.jsonl' và 'val.jsonl':")
    files.upload()

assert os.path.exists('train.jsonl'), 'Thiếu file train.jsonl!'
assert os.path.exists('val.jsonl'), 'Thiếu file val.jsonl!'
print(">>> Đã nhận đủ dữ liệu train.jsonl và val.jsonl thành công!")


## Bước 3: Đọc dữ liệu & Kiểm tra cấu trúc phân tầng

In [ ]:
import json
from datasets import Dataset

def load_jsonl_to_dataset(file_path):
    conversations = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                conversations.append(json.loads(line))
    return Dataset.from_list(conversations)

train_dataset = load_jsonl_to_dataset('train.jsonl')
val_dataset = load_jsonl_to_dataset('val.jsonl')

print(f"Số lượng mẫu tập Huấn luyện (Train): {len(train_dataset)} mẫu (80%)")
print(f"Số lượng mẫu tập Kiểm định (Val)   : {len(val_dataset)} mẫu (10%)")
print("\nMẫu đầu tiên tập Train:")
print("System Prompt:", train_dataset[0]['messages'][0]['content'])
print("Question     :", train_dataset[0]['messages'][1]['content'][:120], "...")


## Bước 4: Tải Base Model (`vilm/vinallama-7b-chat`) ở chế độ 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "vilm/vinallama-7b-chat"

# Cấu hình 4-bit Quantization chuẩn cho GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Đang tải Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Đang tải Base Model vào GPU (mất khoảng 2-3 phút)...\n")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
print(">>> Tải mô hình thành công! Bộ nhớ VRAM đang sử dụng:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## Bước 5: Cấu hình LoRA Adapters (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,                         # LoRA rank
    lora_alpha=32,                # Hệ số co giãn trọng số
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## Bước 6: Huấn luyện Mô hình với Trainer chuẩn của Hugging Face (3 Epochs)
- Sử dụng `transformers.Trainer` và `DataCollatorForSeq2Seq` chính thức từ Hugging Face (ổn định tuyệt đối, không phụ thuộc SFTTrainer của TRL đang bị lỗi Fused LM Head).
- Đã bật `gradient_checkpointing_kwargs={'use_reentrant': False}` để dập tắt hoàn toàn UserWarning của PyTorch.
- Dọn sạch hoàn toàn các monkey-patch forward của TRL.

In [ ]:
import os, shutil, warnings, torch, gc
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# 1. Chống phân mảnh VRAM theo khuyến nghị chính thức của PyTorch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
warnings.filterwarnings('ignore')

# 2. Dọn sạch bộ nhớ đệm và giải phóng các biến trainer cũ
if 'trainer' in locals() or 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

# 3. Khôi phục hoàn toàn forward method nguyên bản của PyTorch/PEFT
for m in [model, getattr(model, 'base_model', None), getattr(getattr(model, 'base_model', None), 'model', None)]:
    if m is not None and 'forward' in m.__dict__:
        del m.forward

# 4. Tiền xử lý dữ liệu và tokenize chuẩn theo Chat Template của Vinallama
# Giới hạn max_length=1536 bao phủ 100% dữ liệu (mẫu dài nhất chỉ ~1400 tokens), triệt tiêu OOM và tăng tốc train
def preprocess_function(examples):
    texts = [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        for conv in examples['messages']
    ]
    tokenized = tokenizer(texts, max_length=1536, truncation=True)
    tokenized['labels'] = [ids.copy() for ids in tokenized['input_ids']]
    return tokenized

print("Đang tokenize tập Train và Val theo Vinallama Chat Template...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=['messages'])
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=['messages'])

# 5. Data Collator chuẩn của Transformers tự động dynamic padding và gán nhãn padding = -100
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    return_tensors='pt',
    padding=True,
    label_pad_token_id=-100
)

# 6. Cấu hình TrainingArguments tối ưu hóa VRAM tuyệt đối cho GPU Tesla T4
training_args = TrainingArguments(
    output_dir="./traffic_law_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,          # Đưa về 1 (thay vì mặc định 8) để triệt tiêu hoàn toàn OOM ở cuối Epoch!
    gradient_accumulation_steps=4,         # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    prediction_loss_only=True,             # Không lưu trữ logits cồng kềnh trong quá trình đánh giá val_dataset
    eval_accumulation_steps=1,             # Giải phóng bộ nhớ đệm ngay trong quá trình eval
    dataloader_pin_memory=False,           # Tiết kiệm bộ nhớ ghim
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none"
)

# 7. Khởi tạo Trainer chính thức của Hugging Face
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

print("Bắt đầu huấn luyện mô hình (3 Epochs)...\n")
trainer.train()

# 7. Lưu thư mục Adapter LoRA
adapter_dir = "./vinallama_traffic_law_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Nén sẵn file zip dự phòng
shutil.make_archive("vinallama_traffic_law_adapter", "zip", adapter_dir)
print("\n>>> Huấn luyện hoàn tất! Đã lưu Adapter và nén file vinallama_traffic_law_adapter.zip an toàn!")


## Bước 7: Kiểm tra suy luận thử nghiệm (Inference Test)
Đánh giá ngay khả năng viện dẫn Luật TTATGTĐB 2024 và Nghị định 168/2024/NĐ-CP của mô hình vừa fine-tune.

In [ ]:
test_question = "Người điều khiển xe ô tô lùi xe trên đường cao tốc bị xử phạt như thế nào theo Nghị định 168/2024/NĐ-CP?"

messages = [
    {"role": "system", "content": "Bạn là Trợ lý AI Tư vấn Pháp luật An toàn Giao thông đường bộ tại Việt Nam. Hãy trả lời câu hỏi của người dùng một cách chính xác, đúng căn cứ pháp lý hiện hành, rõ ràng và chuẩn mực."},
    {"role": "user", "content": test_question}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
model.config.use_cache = True
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== CÂU HỎI THỬ NGHIỆM ===\n", test_question)
print("\n=== CÂU TRẢ LỜI CỦA MÔ HÌNH VỪA FINE-TUNE ===\n", response)


## Bước 8: Hợp nhất (Merge) LoRA Adapter vào Mô hình gốc
- Thiết lập `device_map={"": 0}` để GPU T4 chứa trọn vẹn cả 32 tầng trên VRAM.
- Không offload sang CPU/meta device, dập tắt hoàn toàn các cảnh báo sao chép.

In [ ]:
import gc, os, sys, warnings, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers.modeling_utils as modeling_utils
from peft import PeftModel

warnings.filterwarnings('ignore')

# 1. Vô hiệu hóa caching_allocator_warmup (bỏ qua bước pre-allocate giả lập 12.56GB của Transformers gây OOM)
modeling_utils.caching_allocator_warmup = lambda *args, **kwargs: None

# 2. Ép giải phóng triệt để 100% VRAM của mô hình cũ từ Bước 6 & 7 (xóa sạch cả tensor con và bộ đệm IPython)
if 'model' in globals():
    try:
        for p in model.parameters(): p.data = torch.empty(0)
        for m in model.modules():
            if hasattr(m, 'weight'): m.weight = None
            if hasattr(m, 'bias') and m.bias is not None: m.bias = None
    except Exception: pass

for var in ['model', 'trainer', 'base_model', 'merged_model', 'inputs', 'outputs', 'tokenized_train', 'tokenized_val']:
    if var in globals(): del globals()[var]
    if var in locals(): del locals()[var]

if hasattr(sys.modules.get('__main__'), '_'): del sys.modules.get('__main__')._
if 'Out' in globals(): Out.clear()

gc.collect()
torch.cuda.empty_cache()

vram_free = torch.cuda.mem_get_info()[0] / (1024**3)
print(f"VRAM khả dụng sau khi dọn sạch: {vram_free:.2f} GB / 14.56 GB")

model_id = "vilm/vinallama-7b-chat"
adapter_path = "./vinallama_traffic_law_adapter"
merged_path = "./vinallama_traffic_law_merged"

print("Đang nạp Base Model vào GPU VRAM (nạp tuần tự từng shard vào GPU 0)...\n")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},              # Nạp 100% lên GPU số 0, không dùng meta device
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

print("Đang hợp nhất LoRA Adapter vào mô hình gốc...")
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()
print("Đang lưu mô hình hoàn chỉnh ra đĩa...")
merged_model.save_pretrained(merged_path, max_shard_size="4GB", safe_serialization=True)
tokenizer.save_pretrained(merged_path)

# Giải phóng VRAM để nhường chỗ cho khâu Quantize
del base_model
del merged_model
gc.collect()
torch.cuda.empty_cache()

print(">>> MERGE HOÀN TẤT THÀNH CÔNG 100% CẢ 32 TẦNG!")


## Bước 9: Chuyển đổi sang GGUF (Q4_K_M) bằng `llama.cpp`
Tạo file định danh chuẩn xác cho đồ án: **`vinallama-7b-chat-traffic-law.Q4_K_M.gguf`**.

In [ ]:
import os, sys, subprocess, json, urllib.request, tarfile

# 1. Cài đặt thư viện gguf trực tiếp
print("1/4. Cài đặt thư viện gguf...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gguf"])

# 2. Lấy công cụ llama-quantize (Ưu tiên tải bản pre-built Ubuntu x64 siêu tốc, không tốn RAM)
print("\n2/4. Đang nạp công cụ llama-quantize...")
quantize_path = None

# Thử tải binary build sẵn từ GitHub release chính thức của ggml-org/llama.cpp
try:
    api_url = "https://api.github.com/repos/ggml-org/llama.cpp/releases/latest"
    req = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        rel = json.loads(resp.read().decode())
    for asset in rel.get("assets", []):
        name = asset.get("name", "")
        if "bin-ubuntu-x64.tar.gz" in name and "vulkan" not in name and "cuda" not in name:
            print(f"Đang tải bản pre-built chính thức ({name})...")
            urllib.request.urlretrieve(asset["browser_download_url"], "llama_bin.tar.gz")
            os.makedirs("llama_bin", exist_ok=True)
            with tarfile.open("llama_bin.tar.gz", "r:gz") as t:
                t.extractall("llama_bin")
            break
except Exception as e:
    print(f"(Tải pre-built: {e})")

# Tìm binary llama-quantize trong thư mục vừa giải nén hoặc hệ thống
for root, dirs, files in os.walk("."):
    for f in files:
        if f == "llama-quantize":
            p = os.path.join(root, f)
            os.chmod(p, 0o755)
            quantize_path = p
            break
    if quantize_path: break

# Nếu chưa có pre-built, fallback biên dịch an toàn với 1 tiến trình (-j 1 để chống tràn RAM)
if not quantize_path:
    print("Biên dịch an toàn llama-quantize (-j 1 để chống OOM RAM)...\n")
    if not os.path.exists("llama.cpp"): subprocess.run(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp.git"])
    subprocess.run("cmake -B llama.cpp/build -S llama.cpp -DGGML_CUDA=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null", shell=True)
    proc_b = subprocess.Popen("cmake --build llama.cpp/build --config Release -j 1 --target llama-quantize", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc_b.stdout: print(line, end="", flush=True)
    proc_b.wait()
    for root, dirs, files in os.walk("llama.cpp/build"):
        for f in files:
            if f == "llama-quantize":
                quantize_path = os.path.join(root, f)
                os.chmod(quantize_path, 0o755)
                break
        if quantize_path: break

print(f">>> Công cụ Quantize đã sẵn sàng tại: {quantize_path}")

# 3. Chuyển đổi safetensors sang GGUF FP16 (Stream log thời gian thực)
print("\n3/4. Đang chuyển đổi safetensors sang GGUF FP16 (Stream log trực tiếp)...")
if not os.path.exists("llama.cpp"): subprocess.run(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp.git"])
cmd_convert = [sys.executable, "-u", "llama.cpp/convert_hf_to_gguf.py", "./vinallama_traffic_law_merged", "--outfile", "./vinallama_traffic_law_fp16.gguf", "--outtype", "f16"]
proc_c = subprocess.Popen(cmd_convert, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_c.stdout: print(line, end="", flush=True)
proc_c.wait()
assert proc_c.returncode == 0, f"Lỗi convert FP16 (code {proc_c.returncode})"

# 4. Lượng tử hóa sang Q4_K_M (Stream log thời gian thực)
print("\n4/4. Đang lượng tử hóa (Quantize) sang Q4_K_M (Stream log trực tiếp)...")
cmd_quant = [quantize_path, "./vinallama_traffic_law_fp16.gguf", "./vinallama-7b-chat-traffic-law.Q4_K_M.gguf", "Q4_K_M"]
proc_q = subprocess.Popen(cmd_quant, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc_q.stdout: print(line, end="", flush=True)
proc_q.wait()
assert proc_q.returncode == 0, f"Lỗi quantize Q4_K_M (code {proc_q.returncode})"

# 5. Dọn dẹp file tạm và kiểm tra kết quả cuối cùng
if os.path.exists("vinallama-7b-chat-traffic-law.Q4_K_M.gguf"):
    if os.path.exists("./vinallama_traffic_law_fp16.gguf"): os.remove("./vinallama_traffic_law_fp16.gguf")
    size_gb = os.path.getsize("vinallama-7b-chat-traffic-law.Q4_K_M.gguf") / (1024**3)
    print("\n" + "="*60)
    print(f"XUẤT XƯỞNG THÀNH CÔNG: vinallama-7b-chat-traffic-law.Q4_K_M.gguf")
    print(f"Dung lượng file: {size_gb:.2f} GB")
    print("="*60)


## Bước 10: Tự động sao lưu Mô hình & Adapter vào Google Drive
Lưu trữ vĩnh viễn cả file GGUF lẫn file Adapter zip vào Google Drive của bạn.

In [ ]:
import os, shutil
from google.colab import drive

print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')

assert os.path.exists('vinallama-7b-chat-traffic-law.Q4_K_M.gguf'), 'Không tìm thấy file GGUF!'
assert os.path.exists('vinallama_traffic_law_adapter.zip'), 'Không tìm thấy file Adapter zip!'

# Sao chép file GGUF sang Drive
gguf_dest = '/content/drive/MyDrive/vinallama-7b-chat-traffic-law.Q4_K_M.gguf'
shutil.copy('vinallama-7b-chat-traffic-law.Q4_K_M.gguf', gguf_dest)
print(f"1. Đã lưu thành công file GGUF vào: {gguf_dest}")

# Sao chép file Adapter zip dự phòng sang Drive
adapter_dest = '/content/drive/MyDrive/vinallama_traffic_law_adapter.zip'
shutil.copy('vinallama_traffic_law_adapter.zip', adapter_dest)
print(f"2. Đã lưu thành công file Adapter zip vào: {adapter_dest}")

print("\nTOÀN BỘ TIẾN TRÌNH FINE-TUNE ĐÃ HOÀN TẤT!")
